# Classificatie van nieuwsartikelen

In deze notebook gaan we verder werken op de AG-news nieuwsartikelen dataset.
In de vorige notebook hebben we bekeken hoe we tekstuele data kunnen preprocessen.
In deze notebook gaan we classificatie uitvoeren door gebruik te maken van recurrente neurale netwerken.

In [11]:
[1] + [0] * 5

[1, 0, 0, 0, 0, 0]

In [13]:
# Import necessary libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import kagglehub
from sklearn.model_selection import train_test_split
from collections import Counter
import re

MAX_NUM_WORDS = 20000
MAX_SEQUENCE_LENGTH = 50
EMBEDDING_DIM = 60

# Load the dataset
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

def read_csv(filename):
    df = pd.read_csv(filename)
    print(df.columns)
    df['text'] = df['Title'] + " " + df['Description']
    df['label'] = df['Class Index'] -1

    return df[['text', 'label']]

df_train = read_csv(f"{path}/train.csv")
df_test = read_csv(f"{path}/test.csv")

display(df_train.head())

# Tokenizer
def simple_tokenizer(text):
    return re.findall(r"\b\w+\b", text.lower())

# tel hoeveel keer elk woord voorkomt
counter = Counter()
for text in df_train['text']:
    counter.update(simple_tokenizer(text))
print(counter.most_common(5))

# geef de meest voorkomende woorden een cijfer
vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common(MAX_NUM_WORDS))} # +2 voor de twee speciale symbolen
vocab['<unk>'] = 0
vocab['<pad>'] = 1
print('Size of vocab', len(vocab))

# zet elk woord om naar een cijfer
def encode(text, vocab, max_len):
    tokens = simple_tokenizer(text)
    idxs = [vocab.get(token, vocab['<unk>']) for token in tokens] # woord -> getal (onbekende woorden krijgen ook een getal door de defaultwaarde in get)
    idxs = idxs[:max_len] # truncating
    idxs += [vocab['<pad>']] * (max_len - len(idxs)) # padding
    return idxs

num_samples = 1000

# Dataset + dataloader
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        super(TextDataset,self).__init__()
                
        self.texts = [encode(text, vocab, MAX_SEQUENCE_LENGTH) for text in texts]
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_dataset = TextDataset(df_train['text'][:num_samples], df_train['label'][:num_samples])
test_dataset = TextDataset(df_test['text'][:num_samples], df_test['label'][:num_samples])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

texts, labels = next(iter(train_loader))
print(texts.shape, labels.shape)

Index(['Class Index', 'Title', 'Description'], dtype='object')
Index(['Class Index', 'Title', 'Description'], dtype='object')


,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


[('the', 205468), ('to', 120734), ('a', 113341), ('of', 98647), ('in', 96424)]
Size of vocab 20002
torch.Size([1000, 50])
torch.Size([64, 50]) torch.Size([64])


## Opbouwen, trainen en evalueren van een RNN

In [27]:
# RNN model
EMBEDDING_DIM = 60

class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_dim, output_dim):
        super(RNNModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, rnn_dim, num_layers=10, batch_first=True)
        self.fc = nn.Linear(rnn_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        outputs, hidden = self.rnn(x) # 50 outputs (1 per woord), enkel de laatste hidden
        x = hidden[-1] # de hidden bevat de samengevatte informatie van de hele zin
        x = self.fc(x) # classificeer
        return x

model = RNNModel(len(vocab), EMBEDDING_DIM, 1000, 4)
texts, labels = next(iter(train_loader))
outputs = model(texts)

print(texts.shape, labels.shape)
print(outputs.shape)

torch.Size([64, 50]) torch.Size([64])
torch.Size([64, 4])


In [ ]:
# Train het Model

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 25
for epoch in range(num_epochs):
    running_loss = 0
    for texts, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, loss: {running_loss/len(train_loader)}")

Epoch 1/25, loss: 1.8894349709153175
Epoch 2/25, loss: 1.4364058449864388
Epoch 3/25, loss: 1.3950795531272888


In [ ]:
# Evalueer het Model

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for texts, labels in test_loader:
        outputs = model(texts)
        _, predictions = torch.max(outputs, axis=1)

        total += len(labels)
        correct += (predictions == labels).sum().item()

print(f"Accuraatheid is {correct/total}")

## Oefeningen

* Voeg een extra Dense-laag toe na de RNN-laag. Experimenteer met het aantal neuronen in deze laag en analyseer hoe de prestaties veranderen.
* Pas het model aan om in plaats van een SimpleRNN-laag een LSTM of GRU-laag te gebruiken. Vergelijk de prestaties van de drie typen recurrente netwerken.

In [ ]:
# Oefening 1

In [ ]:
# Oefening 2

**Oefening 3**

Volg de tutorial op de volgende link: https://www.tensorflow.org/text/tutorials/text_generation
Werk hieronder het gelijkaardige probleem uit maar maak het door gebruik te maken van pytorch in plaats van tensorflow voor het model op te bouwen.
In deze tutorial wordt er tekst gegenereerd die lijkt op tekst geschreven door shakespeare.
Let op dat dit een vereenvoudigde versie is waarbij karakter per karakter wordt gegenereerd en niet woord per woord. Er is dus geen garantie dat er echte woorden gemaakt worden.

In [ ]:
import keras
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import random

path_to_file = keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text)} characters')
print(text[:250])
# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

# Character to index mapping
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

# TODO: Encodeer elk karakter in tekst naar een nummer, uitkomst is een list ipv een string

# TODO: Maak een dataset aan waarbij de tekst (uit voorgaande todo) omzet naar een reeks sequenties
# input 100 aaneensluitende karakters, output is het karakter erop volgende

# TODO: indien nodig maak een subset tot 10 of 1% van de dataset

# Check a single example
sample_x, sample_y = dataset[0]
print("Input (x):", sample_x)
print("Target (y):", sample_y)
print("Decoded Input:", ''.join(idx_to_char[idx] for idx in sample_x.numpy()))
print("Decoded Target:", idx_to_char[sample_y.item()])
print('Rows', len(dataset))

In [ ]:
# TODO: Maak een rnn model bestaande uit een embedding layer, gru layer en linear layer
# Maak het mogelijk om aan de forward funtie een parameter toe te voegen om ook de hidden state terug te geven en om de hidden state mee te geven voor de gru laag
# 
vocab_size = len(idx_to_char)
print(vocab_size)
embedding_dim = 50
rnn_units = 60

In [ ]:
# test 1 sample om door het model te sturen
# kijk of je dimensies correct aan elkaar gekoppeld zijn

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import os
import math

batch_size = 64
seq_length = 100
epochs = 5

shakespeare = ShakespeareModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

# TODO: train het rnn model

In [ ]:
import torch.nn.functional as F

def generate_text(model, start_string, char_to_idx, idx_to_char, vocab_size, generation_length=100, temperature=1.0):
    model.eval()  # Set model to evaluation mode
    
    # Convert start_string to indices
    input_indices = torch.tensor([char_to_idx[char] for char in start_string], dtype=torch.long).unsqueeze(0)
    
    generated_text = start_string
    states = None  # Initial state (None means it will be initialized automatically)
    
    for _ in range(generation_length):
        # Genereer opeenvolgend nieuwe tokens
        pass
    
    return generated_text


In [ ]:
# Example start string and generation parameters
start_string = "ROMEO: "
generation_length = 200
temperature = 0.8

# Generate text
generated_text = generate_text(
    model=shakespeare,
    start_string=start_string,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    vocab_size=vocab_size,
    generation_length=generation_length,
    temperature=temperature
)

print("Generated Text:")
print(generated_text)
